<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC Spot Grid Trading Backtest

**Workflow:** Setup → Historical Data → Excel Grid Model → Historical Grid Configuration → Backtest Engine → Results

## 1. Setup

In [ ]:
import os
import io
import glob
import datetime

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/03.Trading/00.Live Trading'
print(f'Data directory: {DATA_DIR}')

## 2. Configuration

In [ ]:
# Market data configuration
SYMBOL = 'BTCUSDT'
START_DATE = '2024-01-01'
END_DATE = '2026-01-01'
TIMEFRAMES = ['1m', '1h', '1d']

# KZM Excel-template parameters used only for replication check
GRID_CAPITAL = 3000.0
GRID_CEILING = 8987.0
GRID_FLOOR = 1987.0
GRID_GAP = 70.0

# Trading fees
BUY_FEE = 0.001
SELL_FEE = 0.001

## 3. Historical Data

The first backtest version uses **BTCUSDT 1-minute OHLCV** as the execution data.

In [ ]:
def download_binance_history(symbol, start_date_str, end_date_str, timeframes, data_dir):
    start_date = datetime.datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date = datetime.datetime.strptime(end_date_str, '%Y-%m-%d')
    os.makedirs(data_dir, exist_ok=True)
    current_date = start_date

    while current_date < end_date:
        year, month = current_date.year, current_date.month
        for timeframe in timeframes:
            url = (
                'https://data.binance.vision/data/spot/monthly/klines/'
                f'{symbol}/{timeframe}/{symbol}-{timeframe}-{year}-{month:02d}.zip'
            )
            try:
                print(f'Downloading {symbol} {timeframe} for {year}-{month:02d}...')
                response = requests.get(url, timeout=60)
                response.raise_for_status()
                columns = [
                    'open_time', 'open', 'high', 'low', 'close', 'volume',
                    'close_time', 'quote_volume', 'number_of_trades',
                    'taker_buy_base', 'taker_buy_quote', 'ignore'
                ]
                df_month = pd.read_csv(
                    io.BytesIO(response.content), compression='zip',
                    header=None, names=columns
                )
                time_unit = 'us' if year >= 2025 else 'ms'
                df_month['open_time'] = pd.to_datetime(df_month['open_time'], unit=time_unit, utc=True)
                df_month['close_time'] = pd.to_datetime(df_month['close_time'], unit=time_unit, utc=True)
                numeric_cols = ['open', 'high', 'low', 'close', 'volume']
                df_month[numeric_cols] = df_month[numeric_cols].astype(float)
                save_path = os.path.join(data_dir, f'{symbol}-{timeframe}-{year}-{month:02d}.csv')
                df_month.to_csv(save_path, index=False)
                print(f'Saved: {save_path}')
            except Exception as exc:
                print(f'Could not download {symbol} {timeframe} {year}-{month:02d}: {exc}')

        if current_date.month == 12:
            current_date = current_date.replace(year=current_date.year + 1, month=1)
        else:
            current_date = current_date.replace(month=current_date.month + 1)

In [ ]:
def combine_monthly_csv(symbol, timeframes, data_dir):
    output_paths = {}
    for timeframe in timeframes:
        pattern = os.path.join(data_dir, f'{symbol}-{timeframe}-????-??.csv')
        file_list = sorted(glob.glob(pattern))
        if not file_list:
            print(f'No monthly files found for timeframe: {timeframe}')
            continue
        combined = pd.concat([pd.read_csv(path) for path in file_list], ignore_index=True)
        combined['open_time'] = pd.to_datetime(combined['open_time'], utc=True)
        combined = (combined.drop_duplicates(subset='open_time')
                    .sort_values('open_time').reset_index(drop=True))
        output_path = os.path.join(data_dir, f'{symbol}-{timeframe}-combined.csv')
        combined.to_csv(output_path, index=False)
        output_paths[timeframe] = output_path
        print(f'Created: {output_path} ({len(combined):,} rows)')
    return output_paths

In [ ]:
def load_market_data(symbol, timeframe, data_dir):
    file_path = os.path.join(data_dir, f'{symbol}-{timeframe}-combined.csv')
    if not os.path.exists(file_path):
        raise FileNotFoundError(f'Combined data file not found: {file_path}')

    df = pd.read_csv(file_path)
    df['open_time'] = pd.to_datetime(df['open_time'], utc=True)
    numeric_cols = ['open', 'high', 'low', 'close', 'volume']
    df[numeric_cols] = df[numeric_cols].astype(float)
    return (df.drop_duplicates(subset='open_time')
              .sort_values('open_time')
              .reset_index(drop=True))

# Primary execution dataset
df_1m = load_market_data(SYMBOL, '1m', DATA_DIR)
df_1m.head()

In [ ]:
def validate_market_data(df):
    ohlc_cols = ['open', 'high', 'low', 'close']
    print(f'Rows              : {len(df):,}')
    print(f'Start             : {df["open_time"].min()}')
    print(f'End               : {df["open_time"].max()}')
    print(f'Duplicate times   : {df["open_time"].duplicated().sum():,}')
    print(f'Rows missing OHLC : {df[ohlc_cols].isna().any(axis=1).sum():,}')

validate_market_data(df_1m)

## 4. Excel Grid Model

This section is only a baseline check against the KZM Excel template before historical execution is added.

In [ ]:
def build_excel_grid_table(capital, ceiling, floor, gap, buy_fee=0.001, sell_fee=0.001):
    if capital <= 0:
        raise ValueError('capital must be greater than 0.')
    if ceiling <= floor:
        raise ValueError('ceiling must be greater than floor.')
    if gap <= 0:
        raise ValueError('gap must be greater than 0.')

    raw_levels = (ceiling - floor) / gap
    if not np.isclose(raw_levels, round(raw_levels)):
        raise ValueError('(ceiling - floor) must be exactly divisible by gap.')

    n_levels = int(round(raw_levels))
    capital_per_level = capital / n_levels
    buy_prices = ceiling - gap * np.arange(1, n_levels + 1)
    sell_prices = buy_prices + gap

    gross_base_amount = capital_per_level / buy_prices
    buy_fee_base = gross_base_amount * buy_fee
    base_amount = gross_base_amount - buy_fee_base
    gross_sell = base_amount * sell_prices
    sell_fee_quote = gross_sell * sell_fee
    net_sell = gross_sell - sell_fee_quote
    profit = net_sell - capital_per_level

    return pd.DataFrame({
        'level': np.arange(1, n_levels + 1),
        'buy_price': buy_prices,
        'sell_price': sell_prices,
        'capital_per_level': capital_per_level,
        'gross_base_amount': gross_base_amount,
        'buy_fee_base': buy_fee_base,
        'base_amount': base_amount,
        'gross_sell': gross_sell,
        'sell_fee_quote': sell_fee_quote,
        'net_sell': net_sell,
        'profit': profit
    })

df_grid_excel = build_excel_grid_table(
    GRID_CAPITAL, GRID_CEILING, GRID_FLOOR, GRID_GAP, BUY_FEE, SELL_FEE
)
df_grid_excel.head()

In [ ]:
# Checkpoint: first Excel pair = 8,917 -> 8,987
excel_check = df_grid_excel.iloc[0]
print(f'Buy price  : {excel_check["buy_price"]:.2f}')
print(f'Sell price : {excel_check["sell_price"]:.2f}')
print(f'Base amount: {excel_check["base_amount"]:.9f}')
print(f'Profit     : {excel_check["profit"]:.6f} USDT')

EXPECTED_FIRST_PROFIT = 0.175064
assert np.isclose(excel_check['profit'], EXPECTED_FIRST_PROFIT, atol=1e-6), 'Excel replication checkpoint failed.'
print('Excel replication checkpoint: PASSED')

## 5. Historical Grid Configuration — V0

For the first execution test, the arithmetic grid is derived from the **full BTCUSDT 1-minute historical range**.

**Important:** this uses future information from the same test period, so V0 contains look-ahead bias. It is for validating execution logic only, not for reporting strategy performance.

In [ ]:
BACKTEST_CAPITAL = 3000.0
TARGET_GRID_LEVELS = 100
PRICE_ROUNDING = 1000.0

historical_low = df_1m['low'].min()
historical_high = df_1m['high'].max()

# Expand boundaries outward to clean 1,000-USDT levels
BACKTEST_FLOOR = np.floor(historical_low / PRICE_ROUNDING) * PRICE_ROUNDING
BACKTEST_CEILING = np.ceil(historical_high / PRICE_ROUNDING) * PRICE_ROUNDING

# Exactly 100 arithmetic intervals
BACKTEST_GAP = (BACKTEST_CEILING - BACKTEST_FLOOR) / TARGET_GRID_LEVELS
CAPITAL_PER_LEVEL = BACKTEST_CAPITAL / TARGET_GRID_LEVELS

print('===== V0 Backtest Grid Configuration =====')
print(f'Historical Low     : {historical_low:,.2f} USDT')
print(f'Historical High    : {historical_high:,.2f} USDT')
print(f'Grid Floor         : {BACKTEST_FLOOR:,.2f} USDT')
print(f'Grid Ceiling       : {BACKTEST_CEILING:,.2f} USDT')
print(f'Number of Grids    : {TARGET_GRID_LEVELS}')
print(f'Grid Gap           : {BACKTEST_GAP:,.2f} USDT')
print(f'Capital / Grid     : {CAPITAL_PER_LEVEL:,.2f} USDT')
print(f'Buy Fee            : {BUY_FEE:.3%}')
print(f'Sell Fee           : {SELL_FEE:.3%}')

In [ ]:
def build_backtest_grid(capital, ceiling, floor, n_levels, buy_fee=0.001, sell_fee=0.001):
    if n_levels <= 0:
        raise ValueError('n_levels must be greater than 0.')
    gap = (ceiling - floor) / n_levels
    return build_excel_grid_table(capital, ceiling, floor, gap, buy_fee, sell_fee)

df_grid_backtest = build_backtest_grid(
    BACKTEST_CAPITAL,
    BACKTEST_CEILING,
    BACKTEST_FLOOR,
    TARGET_GRID_LEVELS,
    BUY_FEE,
    SELL_FEE
)

df_grid_backtest.head()

## 6. Backtest Engine

Execution timeframe: **1 minute**.

Baseline rules for the next step:
- initial portfolio = 100% USDT,
- one open position maximum per grid interval,
- buy when a 1-minute candle touches/crosses a buy level,
- sell one grid above the buy level,
- multiple levels may fill in one minute if cash is sufficient,
- a position opened in a candle cannot close in that same candle,
- buy fee is deducted from BTC received,
- sell fee is deducted from USDT proceeds.

The execution loop will be implemented next.

## 7. Trade Log & Performance

Planned outputs: completed grid cycles, fees, realized profit, open inventory, cash, mark-to-market portfolio value, net return, maximum drawdown, and Calmar ratio.